In [ ]:
# импортируем нужные библиотеки

# для работы с файлами формата .json
import json
# для упрощения отправки HTTP-запросов
import requests
# для генерации случайных чисел и строк (лучше, чем random)
import secrets
# Natural Language Toolkit (для получения слова)
import nltk
# для работы с временем
import time

In [ ]:
# эта функция реализует поиск определения слова в словаре Merriam-Webster

def get_definition(url):
    
    # вводим api-ключ
    api_key = 'e2f920d8-5454-43e7-befa-fe7fcaec4212'
    
    # получаем URL с помощью API
    try:
        url = url + "?key=" + api_key
        # отправляем запрос
        requested_page = requests.get(url, timeout=5)
        
        # проверяем статус страницы
        if requested_page.status_code != 200:
            return 'No Internet'

    except:
        return 'No Internet'
    
    # на случай, если слова нет в словаре
    try:
        page = requested_page.json()
    except:
        return None

    # если вернулся не список или список пуст, или первый элемент является строкой, пропускаем
    if not isinstance(page, list) or len(page) == 0 or isinstance(page[0], str):
        return None

    # безопасное извлечение определения
    try:
        response = page[0]['shortdef'][0]
        return response

    except (KeyError, IndexError, TypeError, AttributeError) as e:
        return None
    
    return response

In [ ]:
# для работы с nltk следует предварительно установить  библиотеку в системе: pip install nltk

# скачиваем и импортируем wordnet
nltk.download('wordnet')
from nltk.corpus import wordnet as wn

# загружаем существительные
noun_synsets = list(wn.all_synsets('n'))
valid_nouns = []

def get_nouns():
    
    try:
        for syn in noun_synsets:
            for lemma in syn.lemmas():
                name = lemma.name()
        # применяем фильтр длины слова, наличия пробелов и дефисов
                if len(name) > 6 and '_' not in name and '-' not in name and name.islower():
                # добавляем в список
                    valid_nouns.append(name)
        return valid_nouns
    
    except:
        return "No Internet"

In [ ]:
# создаём пару "слово-определение"
def get_random_pair():

    # пустой словарь для будущей пары
    random_pair = {}

    valid_nouns = get_nouns()
    
    # на случай отсутствия соединения
    if valid_nouns == "No Internet":
        return "No Internet"

    # получаем случайный элемент списка всех слов
    else:
        random_noun = secrets.choice(valid_nouns)

    # если получено пустое слово
    if not random_noun or random_noun.strip() == "": 
        return None
    
    # создаём адрес запроса
    url  = "https://www.dictionaryapi.com/api/v3/references/collegiate/json/" + random_noun
    # получаем определение
    response = get_definition(url)
    
    # если нет связи
    if response == 'No Internet':
        return "No Internet"
    
    # получаем новые слова, пока не придёт непустой ответ
    while response == 'ul' or response == None or response == 'None' or '\\' in response:
        
        random_noun = secrets.choice(valid_nouns)
        # формируем новый url
        url  = "https://www.dictionaryapi.com/api/v3/references/collegiate/json/" + random_noun
        # ждём, чтобы не перегружать api
        time.sleep(0.1)
        # получаем определение
        response = get_definition(url)
        
        # если нет связи
        if response == "No Internet":
            return "No Internet."
    
    # заполняем словарь, убирая лишнее            
    random_pair[random_noun] = response.strip()

    return random_pair

In [ ]:
# функция для выбора режима игры: через интернет или через базу 
def choice_of_game(choice):

    # если выбрана игра через базу
    if choice == '1':
        # открываем базу, выбираем случайный ключ (слово), его значение (определение) и формируем словарь для пары
        with open('base_installed.json', 'r', encoding='utf-8') as base:
            base = json.load(base)
            random_key = secrets.choice(list(base.keys()))
            random_pair = {random_key: base[random_key]}
            
        return random_pair
    
    # если выбрана игра через интернет
    elif choice == '2':
        # если нет связи, выбираем пару из базы
        if get_random_pair() == 'No Internet':
            print('Connection problem. Choosing word from the base...')
            
            with open('base_installed.json', 'r', encoding='utf-8') as base:
                base = json.load(base)
                random_key = secrets.choice(list(base.keys()))
                random_pair = {random_key: base[random_key]}
                
            return random_pair
        
        # если связь в порядке, формируем пару через уже написанные функции, пока не получим непустую 
        else:
            random_pair = get_random_pair()
            
            while not random_pair:
                random_pair = get_random_pair()
                
            return random_pair

In [ ]:
# определяем форму слова attempt для вывода на экран
def attemps_left(num):
    
    if num == 1:  
        return 'attempt'
    
    else:
        return 'attempts'

In [ ]:
# функция для проверки, что введён допустимый символ
def check_symbol(letter, used_letters):
    while True:
            
        if letter.lower() == '' or letter.lower() == ' ':
            letter = input('Please enter something: \n>>> ').lower()
            continue
                
        if letter.lower() in used_letters:
            letter = input('You have already entered this letter. Please try again: \n>>> ').lower()
            continue
            
        if len(letter) > 1:
            letter = input('Please enter one letter: \n>>> ').lower()
            continue
            
        if letter.lower() not in alphabet:
            letter = input('Please enter an English letter: \n>>> ').lower()
            continue
            
        return letter

In [ ]:
def game_start():
    
    choice = \
    input('How do you want to play? \n 1. With a saved base (5000 words). \n 2. Via the Internet. \n My choice: \n>>> ')
    
    # если выбран несуществующий вариант, предлагаем выбор, пока не будет выбран существующий
    while choice not in ['1', '2']:
        choice = input('Enter 1 or 2: \n>>> ')
        
    return choice

In [ ]:
# функция для вывода слова и определения + выбора продолжения сценария в конце игры
def game_final(game_dict):
    
    #выводим определение слова
    for key in game_dict:
        print(key.upper(), ': ', game_dict[key], sep='', end='\n')
                
        # можно посмотреть угаданные слова
        if(input('To show guessed words, enter ''g'', to continue, enter any other '
                             'symbol'' \n>>>')) == 'g':
            print(*guessed_words, sep='\n')
                    
        # предлагаем сыграть ещё раз, если да, сбрасываем списки букв
        conscent = input('Wanna play once more? To agree enter "+", to refuse enter anything else: \n>>> ')
        if conscent == '+':
            guessed_letters = []
            used_letters = []
                               
        # в противном случае возвращаем 'end'   
        else:
            return 'end'

In [ ]:
# допустимые символы: строчные буквы английского алфавита 
alphabet = 'abcdefghijklmnopqrstuvwxyz'

# создаём пустой список для угаданных слов
guessed_words = []

# основная игровая функция

def play(guessed_words):
    
    # выбираем режим игры
    choice = input('How do you want to play? \n 1. With a saved base (5000 words). \n 2. Via the Internet. \n My choice: \n>>> ')
    
    # если выбран несуществующий вариант, предлагаем выбор, пока не будет выбран существующий
    while choice not in ['1', '2']:
        choice = input('Enter 1 or 2: \n>>> ')
        
    print('Please wait a moment...')
    
    # получаем пару
    game_dict = choice_of_game(choice)
    
    # берём угадываемое слово, т. е. ключ
    for key in game_dict.keys():
        word = key

    # можно прибавить к числу букв слова дополнительные попытки
    count = input('Enter number of attemps to add to the length of the word: \n>>> ')
    
    # если введённый символ не обозначает натуральное число, предлагаем ввод снова
    while count.isdigit() == False:
        count = input('Enter a number not less than zero: \n>>> ')
        
    # число доп. попыток
    count = int(count) + len(word)
    
    # строка, содержащая символы "*" на каждую букву загаданного слова
    word_completion = list('*' * len(word)) 

    # список для уже названных букв
    guessed_letters = []  
    used_letters = []

    #выводим на экран замаскированное слово
    while True:
        print('\n', *word_completion, '\n', sep='')
        
        # ввод буквы
        letter = input('Enter a letter: \n>>>').lower()
        
        #проверяем, что введён допустимый символ
        letter = check_symbol(letter, used_letters)

        # добавляем букву в список использованных
        used_letters.append(letter)
                
        # если угадана буква и не была введена пустая строка
        if letter in word and letter != '':
            
            # заменяем "*" на угаданную букву
            for i in range(len(word_completion)):
                
                if word[i].lower() == letter.lower():
                    word_completion[i] = letter.lower()
            
            # заполняем и выводим список угаданных букв
            guessed_letters.append(letter.lower())
            print('You guessed the letters:', guessed_letters)
            
            #если угадано слово, добавляем его в список, поздравляем пользователя
            if ''.join(word_completion) == word:
                guessed_words.append(''.join(word_completion))
                print('Congrats! You guessed the word ')
                
                # выводим слово, определение, выбираем дальнейший сценарий
                final = game_final(game_dict)
                
                return final
                
                break

        
        # если буквы в слове нет
        else:
            count -= 1
            print('Oops!', count, attemps_left(count), 'left...')
            
            # если попытки закончились, пользователь проиграл; выводим загаданное слово
            if count == 0:
                print('You lost... The hidden word was', end=' ')
                
                # выводим слово, определение, выбираем дальнейший сценарий
                final = game_final(game_dict)
                
                return final
                
                break

In [ ]:
# приветствие
print("Let's play 'Hangman'! All definitions are taken from Merriam-Webster Dictionary.", \
      sep='\n', end='\n')


# играем!
while True:
    
    if play(guessed_words) == 'end':
        guessed_words = []
        print('Thanks for the game!')
        
        break